# Hadita — Gemini 3 → PAGE XML (Transkribus-ready)

Runs **Gemini 3** agentic OCR on a **processed/dewarped** Hadita tax-register page, using the project's tuned prompt rules ported from Approach M (`compare_ocr.py` — column-order warning, `Nature_of_Entry` vocabulary, multi-category and breakdown-row rules, resume-numbering rule, [RED]/[?]/~~strike~~ markers, **"one value per cell — never `X / Y`"**, **"never omit a row"**), and patches the result into the page's existing PAGE XML so it drops straight back into your Transkribus workflow.

**Inputs to upload (for page _N_):**
1. `Hadita-{N}Processed.jpg` — the dewarped processed image
2. `Hadita_{N}.xml` — the existing PAGE XML from `Transkribus upload/final/`

**Output:** `Hadita_{N}_G3.xml` — same geometry as the input XML; only the `<Unicode>` text of each TableCell is overwritten with the Gemini-3 transcription. The prior cell text is dropped (kept in Transkribus version history).

**Model:** `gemini-3.5-flash`, `medium` thinking, `code_execution` (agentic vision) on. One full-page call.


In [ ]:
!pip install -q -U google-genai pydantic pillow

In [ ]:
import os
from google import genai

key = None
try:
    from google.colab import userdata
    key = userdata.get("GEMINI_API_KEY")
except Exception:
    key = os.environ.get("GEMINI_API_KEY")
if not key:
    import getpass
    key = getpass.getpass("Paste your GEMINI_API_KEY: ")

os.environ["GEMINI_API_KEY"] = key
os.environ["GOOGLE_API_KEY"] = key
client = genai.Client()
print("client ready")


In [ ]:
# === Set the page number, then upload BOTH files when prompted ===
PAGE_NUM = 10   # ← edit this for each run

MODEL = "gemini-3.5-flash"      # Gemini 3 family (do NOT downgrade to 2.5)
THINKING = "medium"             # minimal | low | medium | high
RESOLUTION = "high"             # ultra_high | high | medium
                                # → the auto-retry will lower this on "exceeds max allowed size"
MAX_OUTPUT_TOKENS = 32768       # bumped from 16384 (medium-thinking needs more headroom)
                                # → the auto-retry will raise this on JSON truncation

from google.colab import files
print(f"Upload TWO files for page {PAGE_NUM}:")
print(f"  1. Hadita-{PAGE_NUM}Processed.jpg     (the processed/dewarped image)")
print(f"  2. Hadita_{PAGE_NUM}.xml               (existing PAGE XML from Transkribus upload/final/)")
uploaded = files.upload()

IMAGE_PATH = next((f for f in uploaded if f.lower().endswith((".jpg", ".jpeg", ".png"))), None)
XML_PATH   = next((f for f in uploaded if f.lower().endswith(".xml")), None)
assert IMAGE_PATH, "No image file uploaded — expected a .jpg/.jpeg/.png"
assert XML_PATH,   "No XML file uploaded — expected the Hadita_{N}.xml from Transkribus upload/final/"
print(f"\nimage: {IMAGE_PATH}\nxml:   {XML_PATH}\nmodel: {MODEL} × {THINKING}")


In [ ]:
from typing import Any, Literal, Optional
from pydantic import BaseModel, Field

# Column order MUST match patch_xml_text.py:LEFT_COLS so each LedgerRow field
# lines up positionally with cell_r{R}_c{C} in the PAGE XML.
LEFT_COLS = [
    "Serial_No", "Date",
    "Property_recorded_under_Block_No", "Property_recorded_under_Parcel_No",
    "Parcel_Cat_No", "Parcel_Area",
    "Nature_of_Entry", "New_Serial_No",
    "Reference_to_Register_of_Changes_Volume_No",
    "Reference_to_Register_of_Changes_Serial_No",
    "Tax_LP", "Tax_Mils", "Total_Tax_LP", "Total_Tax_Mils",
    "Reference_to_Register_of_Exemptions_Entry_No",
    "Reference_to_Register_of_Exemptions_Amount_LP",
    "Reference_to_Register_of_Exemptions_Amount_Mils",
    "Net_Assessment_LP", "Net_Assessment_Mils",
    "Remarks",
]

class BilingualText(BaseModel):
    arabic: Optional[str] = Field(None, description="Arabic original, transcribed exactly.")
    english: Optional[str] = Field(None, description="(unused — leave null; do NOT translate)")

class LedgerRow(BaseModel):
    """One row of the left tax-register table. Empty cells → "" (never null)."""
    serial_no: Optional[str] = Field(None, description="Serial_No. Eastern Arabic digits exactly as written; \"\" if blank (e.g. breakdown rows).")
    date: Optional[str] = Field(None, description="Date (column 2 — NOT Block_No). Year-like Eastern Arabic digits (e.g. ٩٣٨), or \" for ditto, or \"\" if blank. Exactly ONE value; never compound like 'X / Y'.")
    block_no: Optional[str] = Field(None, description="Property_recorded_under_Block_No. 4-digit Eastern Arabic (range ٤١٣٢–٤١٥٢). \"\" on multi-category continuation rows (never ditto/fabricate).")
    parcel_no: Optional[str] = Field(None, description="Property_recorded_under_Parcel_No. Eastern Arabic digits. \"\" on multi-category continuation rows.")
    cat_no: Optional[str] = Field(None, description="Parcel_Cat_No (cultivation category). Eastern Arabic digits, usually two.")
    area: Optional[str] = Field(None, description="Parcel_Area. Eastern Arabic digits; standardize thousands separator to ASCII comma.")
    nature_of_entry: Optional[str] = Field(None, description="Nature_of_Entry. Raw Arabic + symbols only — NEVER translate, gloss, or interpret. Common: تح, تسلل, شراء, شرائي, بيع, ضريبة حرب. Combinations: 'تح ✓'. Blank → \"\". Ditto → \".")
    new_serial_no: Optional[str] = Field(None, description="New_Serial_No. Eastern OR Western digits; or short cross-ref like 'انظر ٩٧'.")
    ref_volume_no: Optional[str] = Field(None, description="Reference_to_Register_of_Changes_Volume_No. Eastern digits, ✓, '...', English abbreviation (T.D.L), or \"\".")
    ref_serial_no: Optional[str] = Field(None, description="Reference_to_Register_of_Changes_Serial_No. Eastern digits, '...', a Western year paired with T.D.L/T.P.L (e.g. 1940), or \"\".")
    tax_lp: Optional[str] = Field(None, description="Tax — L.P. column. Usually NOT a number (most assessments < 1 LP): '✓' (assessed), '-' (nil), Eastern digit ONLY if tax ≥ 1 LP, \"\" if blank. NEVER copy the Mils value here. NEVER leave blank when ✓ or - is drawn.")
    tax_mils: Optional[str] = Field(None, description="Tax — Mils. Eastern Arabic digits (usually 3 with leading zero, e.g. ٠٨٥), '-' for nil, or \"\".")
    total_tax_lp: Optional[str] = Field(None, description="L.P. under 'Total Tax'. Same value domain as Tax_LP.")
    total_tax_mils: Optional[str] = Field(None, description="Mils under 'Total Tax'. Eastern Arabic digits or \"\". Do NOT collapse Total_Tax_Mils into Tax_Mils.")
    exemption_entry_no: Optional[str] = Field(None, description="Entry_No under 'Reference to Register of Exemptions'. Eastern Arabic digits or \"\".")
    exemption_amount_lp: Optional[str] = Field(None, description="L.P. under exemption amount. ✓ / - / Eastern digit / \"\".")
    exemption_amount_mils: Optional[str] = Field(None, description="Mils under exemption amount. Eastern Arabic digits or \"\".")
    net_assessment_lp: Optional[str] = Field(None, description="L.P. under 'Net Assessment'. ✓ / - / Eastern digit / \"\".")
    net_assessment_mils: Optional[str] = Field(None, description="Mils under 'Net Assessment'. Eastern Arabic digits or \"\".")
    remarks: Optional[str] = Field(None, description="Free-text annotation column. Notes, corrections, marginalia. Use ~~old~~ new for a visible correction; append [RED] if red ink; [?] if uncertain.")
    confidence: Literal["high", "medium", "low"] = Field("high", description="Reviewer-facing row confidence. 'medium' or 'low' whenever code_execution was used to disambiguate any cell in this row.")
    reviewer_note: Optional[str] = Field(None, description="Brief pointer for the human reviewer when confidence != 'high'. Name the cell(s) and the doubt, ≤ 120 chars.")

class SectionRows(BaseModel):
    rows: list[LedgerRow] = Field(description="EVERY row of the left table, top to bottom — data, multi-category continuation, breakdown, sub-total, resumed numbering. NEVER omit a row (blank rows → all fields \"\", confidence 'low').")

def _strict_schema(model: type[BaseModel]) -> dict:
    schema = model.model_json_schema()
    def _t(n: Any) -> None:
        if isinstance(n, dict):
            if n.get("type") == "object" and "properties" in n:
                n["required"] = list(n["properties"].keys())
                n["additionalProperties"] = False
            for v in n.values():
                _t(v)
        elif isinstance(n, list):
            for x in n:
                _t(x)
    _t(schema)
    return schema

# Map LedgerRow → row-dict keyed by LEFT_COLS (positional patching to cell_r{R}_c{C})
_FIELD_TO_COL = {
    "serial_no":          "Serial_No",
    "date":               "Date",
    "block_no":           "Property_recorded_under_Block_No",
    "parcel_no":          "Property_recorded_under_Parcel_No",
    "cat_no":             "Parcel_Cat_No",
    "area":               "Parcel_Area",
    "nature_of_entry":    "Nature_of_Entry",
    "new_serial_no":      "New_Serial_No",
    "ref_volume_no":      "Reference_to_Register_of_Changes_Volume_No",
    "ref_serial_no":      "Reference_to_Register_of_Changes_Serial_No",
    "tax_lp":             "Tax_LP",
    "tax_mils":           "Tax_Mils",
    "total_tax_lp":       "Total_Tax_LP",
    "total_tax_mils":     "Total_Tax_Mils",
    "exemption_entry_no": "Reference_to_Register_of_Exemptions_Entry_No",
    "exemption_amount_lp":"Reference_to_Register_of_Exemptions_Amount_LP",
    "exemption_amount_mils":"Reference_to_Register_of_Exemptions_Amount_Mils",
    "net_assessment_lp":  "Net_Assessment_LP",
    "net_assessment_mils":"Net_Assessment_Mils",
    "remarks":            "Remarks",
}

def _to_sinai_row(row: LedgerRow) -> dict[str, str]:
    d = {col: (getattr(row, f) or "") for f, col in _FIELD_TO_COL.items()}
    if row.reviewer_note and row.confidence != "high":
        d["Remarks"] = (f"[reviewer: {row.reviewer_note}] " + d.get("Remarks", "")).strip()
    return d

def _thought_text(step: Any) -> str:
    return "".join(p.text for p in (getattr(step, "summary", None) or []) if getattr(p, "type", "") == "text")

print(f"schema ready: {len(LedgerRow.model_fields)} fields per row, {len(LEFT_COLS)} XML columns")


In [ ]:
SYSTEM_INSTRUCTION = """<role>
You transcribe rows from British Mandate-era rural land tax registers (Form TR/39) for the village of Hadita (الحديثة). Column headers are pre-printed English; entries are handwritten Arabic with Eastern Arabic numerals (٠١٢٣٤٥٦٧٨٩). The page image is the LEFT TABLE only, already dewarped and cropped — there is no right page.
</role>

<columns>
The LEFT TABLE has 20 columns. In order (right-to-left reading, the model schema preserves this):
   1. Serial_No
   2. Date                    ← year-like (e.g. ٩٣٨), often ditto. NOT Block_No.
   3. Property_recorded_under_Block_No   (4-digit, ٤١٣٢–٤١٥٢)
   4. Property_recorded_under_Parcel_No
   5. Parcel_Cat_No           (cultivation category, usually two digits)
   6. Parcel_Area             (Eastern digits; thousands sep → ASCII comma)
   7. Nature_of_Entry
   8. New_Serial_No
   9. Reference_to_Register_of_Changes_Volume_No
  10. Reference_to_Register_of_Changes_Serial_No
  11. Tax_LP        12. Tax_Mils        13. Total_Tax_LP    14. Total_Tax_Mils
  15. Reference_to_Register_of_Exemptions_Entry_No
  16. Exemption Amount_LP    17. Exemption Amount_Mils
  18. Net_Assessment_LP      19. Net_Assessment_Mils
  20. Remarks (free-text annotation)

COLUMN POSITION IS FIXED. Never shift a value leftward to fill an empty cell. Count columns from the right edge of the table when in doubt.
</columns>

<core_rules>
1. Preserve Arabic script and Eastern Arabic numerals (٠١٢٣٤٥٦٧٨٩) exactly as written. Standardize thousands separator to ASCII comma (١٬٢٠٠ → ١,٢٠٠). Western (0-9) and Eastern (٠-٩) both appear in the same document — output each system exactly as written; never convert.
2. Empty cells → "" (empty string). Never invent values.
3. ONE value per cell. NEVER output compound readings like "X / Y", "٩٤٢ / ٩٤٤", or "٤٢ / ٩٤٠". If two readings seem plausible, pick the more legible and append [?] to flag uncertainty.
4. NEVER omit a row. If a row is entirely blank, output it with all fields "" and confidence "low". Omitting rows breaks the positional patch into the PAGE XML.
5. Uncertainty markers (append to the cell value):
   - ~~old~~          strikethrough only
   - ~~old~~ new      visible correction
   - [?]              uncertain read
   - [RED]            written in red ink
6. For bilingual cells the ARABIC is canonical — transcribe what you see; never align Arabic to a presumed English value.
</core_rules>

<numerals>
Distinguish similar handwritten Eastern Arabic digits carefully:
  ٢ (2)  one small angular hook/curve, compact
  ٣ (3)  two scallops/bumps, wider/more open
  ٤ (4)  open hook facing right
  ٦ (6)  small closed circle/loop
  ٨ (8)  similar shape to ٦ but with a vertical extension or wider top — easy to misread as ٦
  ٠ (0)  small dot, usually smaller than ٦

Documented hard cases on this scribe (zoom in via code_execution before committing):
  - ٢/٣ confusion in the leading digit of `area`, `block_no`, `parcel_no`
  - ٣/٤ and ٤/٦ confusion in `tax_mils` and `total_tax_mils`
  - ٦/٨ confusion in the middle digit of `tax_mils` (e.g. ٠٦٩ vs ٠٨٩)
  - The whole 3-digit `tax_mils` cell can read backwards (١٠٧ vs ٠٧١) when faint — read left-to-right; do NOT permute digits.
</numerals>

<register_priors>
Hadita register (village الحديثة). Use these ONLY to break a genuine tie, NEVER to override a digit you can read:
  Date: years ٩٣٨–٩٤٩ (1938–1949).
  Block_No: ٤١٣٢–٤١٥٢ (4132–4152), a 4-digit number.
</register_priors>

<symbols>
  ✓   Checkmark (U+2713): yes / confirmed / assessed.
  "   Ditto mark (two short parallel vertical ticks, double-comma, or curly quotes): repeat the value from the row above. Output exactly: " (ASCII U+0022).
  -   Horizontal dash (U+002D): nil / exempted / zero.
  ... or ..   Three or two dots: not applicable / no data (common in Reference columns).
  T.D.L / T.P.L / D.L   English abbreviations (Transferred to Different/Previous Location): preserve verbatim.

Shape guide for the easy-to-confuse trio:
  ditto " = two vertical strokes
  ✓       = single angled / curved stroke
  ١       = single straight vertical stroke
</symbols>

<nature_of_entry>
Vocabulary on this register (transcribe Arabic exactly — DO NOT translate, gloss, or interpret):
  تح          (abbreviation for تحديث, "update")
  تسلل        (infiltration)
  من تسلل
  شراء / شرائي (purchase)
  بيع         (sale)
  ضريبة حرب    (war tax)

Combinations: Arabic text + checkmark → "تح ✓".
Blank → "". Ditto → ". Checkmark only → ✓.

DO NOT output English words like "transfer", "assessed", "confirmed ditto", "Ditto", or "checked" in this field. Raw Arabic + symbols only.
</nature_of_entry>

<tax_lp_rule>
Tax_LP, Total_Tax_LP, Exemption Amount_LP, Net_Assessment_LP are USUALLY NOT NUMBERS because most assessments are < 1 LP. Allowed values:
  ✓                  assessed
  -                  nil / exempted
  Eastern digit(s)   only when value ≥ 1 LP (rare)
  ""                 genuinely blank
NEVER copy the Mils value into the LP column. NEVER leave the LP column "" when ✓ or - is drawn in the cell.
</tax_lp_rule>

<multi_category_parcels>
A single physical parcel may span MULTIPLE CONSECUTIVE numbered rows — one per cultivation category. Each such row has its OWN Serial_No (consecutive serials e.g. ٣، ٤، ٥، ٦). What is consistently EMPTY in the continuation rows is **Property_recorded_under_Block_No** and **Property_recorded_under_Parcel_No** — those describe the parcel as a whole, not the individual category. Output "" for those two cells on continuation rows; do NOT fabricate, do NOT ditto. The other columns (Date, Nature_of_Entry, New_Serial_No) vary case by case — ditto, written value, or blank.
</multi_category_parcels>

<unnumbered_rows>
Some rows legitimately have NO Serial_No but still contain content. Typical cases:
  - Tax-year breakdown rows: a year in Date and tax figures in Tax / Total / Net columns.
  - Sub-total / running-total rows: figures only in the totals columns.
  - Carry-forward / amendment rows.

Treat each as its own row. Set Serial_No:"" and fill only the cells that have content. NEVER merge a serial-less row's content into the row above or below. NEVER skip the row.

COLUMN POSITION IS FIXED — in a row with no Serial_No: if the row contains a year (e.g. ٩٣٩, ٩٤٠), it belongs in Date, NOT in Serial_No. Tax/Total figures stay in Tax_LP / Tax_Mils / Total_Tax_LP / Total_Tax_Mils; never collapse Total_Tax_Mils into Tax_Mils.

Example tax-year breakdown row:
  serial_no:"", date:"٩٣٩",
  block_no:"", parcel_no:"", cat_no:"", area:"",
  nature_of_entry:"", new_serial_no:"",
  ref_volume_no:"", ref_serial_no:"",
  tax_lp:"-", tax_mils:"٠٢٢", total_tax_lp:"-", total_tax_mils:"٠٢٢",
  exemption_entry_no:"", exemption_amount_lp:"", exemption_amount_mils:"",
  net_assessment_lp:"", net_assessment_mils:"",
  remarks:"", confidence:"high"
</unnumbered_rows>

<resuming_numbered_rows>
After a sequence of unnumbered breakdown rows (e.g. ٩٣٩، ٩٤٠، ٩٤١، ٩٤٢، ٩٤٣), a row with a fresh handwritten serial number (e.g. ٧, following the earlier 1–6 sequence) is a NUMBERED data row — assign it that Serial_No, with its own Block/Parcel/Cat/Area. Subsequent rows continue the new numbering (٨، ٩، ١٠، ١١، …) until the next break.

A resumed numbered row often also has a fresh Date (e.g. ٩٤٤, ٩٤٨) and a distinctive Nature_of_Entry such as ضريبة حرب (war tax) or بيع (sale). Visually verify each row's Serial_No cell; do NOT keep emitting empty Serial_No just because the previous rows were unnumbered.
</resuming_numbered_rows>

<agentic_vision>
You have a Python sandbox (code_execution). USE IT actively wherever a cell is faint, small, or ambiguous — especially:
  - distinguishing ٢ from ٣ in the leading digit of `area`
  - distinguishing ٤ from ٦ in `tax_mils` and `total_tax_mils`
  - telling a ditto mark " from a checkmark ✓ (shape guide above)
  - reading the resumed-numbering serial in the bottom block

Crop the relevant region at full resolution, examine it, then commit a reading. When you used code_execution on a row, set that row's confidence to "medium" (or "low") and add a short reviewer_note naming the doubtful cells.
</agentic_vision>
"""

PROMPT = (
    "<task>\n"
    "This image is a full LEFT TABLE page from the Hadita tax register, dewarped and cropped. "
    "The printed English column-header band is at the top; handwritten data rows fill the rest. "
    "Transcribe EVERY row top to bottom — numbered data rows, multi-category continuations, "
    "tax-year breakdown rows, sub-totals, and any resumed-numbering block at the bottom. "
    "Map each handwritten cell to its column by horizontal position; column meanings are in the schema. "
    "Return SectionRows with one LedgerRow per physical row. NEVER omit a row.\n"
    "</task>"
)

print(f"system_instruction: {len(SYSTEM_INSTRUCTION):,} chars  ·  prompt: {len(PROMPT):,} chars")


In [ ]:
import time, csv, json
from pathlib import Path
from IPython.display import display, Markdown

# Pricing (USD per 1M tokens, Standard paid tier).
# Thinking billed at output rate, cached input at the cached rate.
PRICING = {
    "gemini-3-flash-preview": {"input": 0.50, "output": 3.00, "cached": 0.05},
    "gemini-3.5-flash":       {"input": 1.50, "output": 9.00, "cached": 0.15},
}

def _estimate_cost(model: str, ud: dict) -> float:
    p = PRICING.get(model)
    if not p:
        return float("nan")
    inp = ud.get("total_input_tokens") or 0
    out = ud.get("total_output_tokens") or 0
    tho = ud.get("total_thought_tokens") or 0
    cac = ud.get("total_cached_tokens") or 0
    billable_input = max(inp - cac, 0)
    return (billable_input * p["input"] + cac * p["cached"] + (out + tho) * p["output"]) / 1e6

def _run_once(resolution: str, max_output_tokens: int):
    """One full-page agentic call. Returns (interaction, elapsed_s)."""
    t0 = time.perf_counter()
    interaction = client.interactions.create(
        model=MODEL,
        system_instruction=SYSTEM_INSTRUCTION,
        input=[{
            "type": "user_input",
            "content": [
                {"type": "image", "uri": target_uri, "mime_type": "image/jpeg", "resolution": resolution},
                {"type": "text",  "text": PROMPT},
            ],
        }],
        tools=[{"type": "code_execution"}],
        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": _strict_schema(SectionRows),
        },
        generation_config={
            "thinking_level": THINKING,
            "thinking_summaries": "auto",
            "max_output_tokens": max_output_tokens,
        },
    )
    return interaction, round(time.perf_counter() - t0, 2)

target_uri = client.files.upload(file=IMAGE_PATH, config={"mime_type": "image/jpeg"}).uri
print(f"uploaded {IMAGE_PATH} → {target_uri}")

import threading

def _run_with_heartbeat(resolution: str, max_output_tokens: int):
    """Run the agentic call on a background thread; print elapsed every 30s
    so you can tell 'still running' apart from 'frozen'. The API has no
    progress signal, so this is a heartbeat — NOT an ETA."""
    done = threading.Event()
    result = {}
    def _worker():
        try:
            result["interaction"], result["elapsed"] = _run_once(resolution, max_output_tokens)
        except Exception as e:
            result["error"] = e
        finally:
            done.set()
    threading.Thread(target=_worker, daemon=True).start()
    print(f"running (resolution={resolution!r}, thinking={THINKING!r}) — heartbeat every 30s …", flush=True)
    t0 = time.perf_counter()
    nudge_at = {600, 900}    # 10 min, 15 min
    nudged = set()
    while not done.wait(timeout=30):
        e = int(time.perf_counter() - t0)
        bar = f"  … {e//60}m{e%60:02d}s elapsed"
        for threshold in nudge_at:
            if e >= threshold and threshold not in nudged:
                bar += f"   ⚠ past {threshold//60} min — most calls finish under 5 min on this config. Consider Stop + lowering THINKING to 'low' or 'minimal'."
                nudged.add(threshold)
                break
        print(bar, flush=True)
    if "error" in result:
        raise result["error"]
    return result["interaction"], result["elapsed"]

from pydantic import ValidationError

# Adaptive retry: TWO distinct failure modes need DIFFERENT remedies.
#   (A) "exceeds the maximum allowed size limit"   = backend response cap reached
#       → lower resolution (fewer image tokens per zoom round)
#   (B) ValidationError "EOF while parsing a string" = JSON output truncated mid-stream
#       → raise max_output_tokens (room for the rest of the JSON)
# Up to 3 attempts. Resolution downgrade path: ultra_high → high → medium → low.
RES_DOWNGRADE = {"ultra_high": "high", "high": "medium", "medium": "low"}
res, max_out = RESOLUTION, MAX_OUTPUT_TOKENS
interaction = elapsed = section = None
for attempt in range(1, 4):
    try:
        interaction, elapsed = _run_with_heartbeat(res, max_out)
    except Exception as e:
        msg = str(e)
        if "exceeds the maximum allowed size" in msg or "image_size" in msg:
            new_res = RES_DOWNGRADE.get(res, "low")
            print(f"⚠ (A) backend size-cap at resolution={res!r}. "
                  f"retrying at resolution={new_res!r}  [attempt {attempt+1}/3] …")
            res = new_res
            continue
        raise

    final_text = getattr(interaction, "output_text", None) or interaction.steps[-1].content[0].text
    try:
        section = SectionRows.model_validate_json(final_text)
        break
    except ValidationError as ve:
        if "EOF while parsing" in str(ve) or "json_invalid" in str(ve):
            new_max = min(max_out * 2, 65536)
            print(f"⚠ (B) JSON truncated at max_output_tokens={max_out} "
                  f"(got {len(final_text):,} chars). "
                  f"retrying at max_output_tokens={new_max}  [attempt {attempt+1}/3] …")
            max_out = new_max
            continue
        raise
else:
    raise RuntimeError(
        f"3 attempts exhausted. last settings: resolution={res!r}, max_output_tokens={max_out}. "
        "Try THINKING='low' (fewer thought tokens, more room for output) or split the page."
    )

rows = [_to_sinai_row(r) for r in section.rows]

zoom_rounds = sum(1 for s in interaction.steps if getattr(s, "type", "") == "code_execution_call")
u = getattr(interaction, "usage", None)
ud = {}
if u is not None:
    try: ud = u.model_dump()
    except Exception: ud = {}
cost = _estimate_cost(MODEL, ud)

print(f"\n{len(rows)} rows · {zoom_rounds} zoom rounds · {elapsed}s · ${cost:.4f}")
print("usage:", {k: ud.get(k) for k in ("total_input_tokens","total_output_tokens","total_thought_tokens","total_tool_use_tokens","total_tokens")})

# Append per-page usage to a running CSV (downloaded with the other outputs).
RUNS_CSV = "g3_runs.csv"
_new = not Path(RUNS_CSV).exists()
with open(RUNS_CSV, "a", newline="") as f:
    w = csv.writer(f)
    if _new:
        w.writerow(["page","model","thinking","resolution","max_output_tok","rows","zoom_rounds",
                    "elapsed_s","input_tok","cached_tok","output_tok","thought_tok",
                    "tool_use_tok","total_tok","cost_usd"])
    w.writerow([PAGE_NUM, MODEL, THINKING, res, max_out, len(rows), zoom_rounds, elapsed,
                ud.get("total_input_tokens") or 0, ud.get("total_cached_tokens") or 0,
                ud.get("total_output_tokens") or 0, ud.get("total_thought_tokens") or 0,
                ud.get("total_tool_use_tokens") or 0, ud.get("total_tokens") or 0,
                round(cost, 6)])
print(f"appended to {RUNS_CSV}")

# Save the raw rows alongside the future XML output
ROWS_JSON_PATH = f"Hadita_{PAGE_NUM}_G3.json"
Path(ROWS_JSON_PATH).write_text(json.dumps(rows, ensure_ascii=False, indent=2))
print(f"wrote {ROWS_JSON_PATH}")

# Quick preview — key columns
display(Markdown(f"### Preview — page {PAGE_NUM}, {len(rows)} rows"))
preview_cols = ["Serial_No","Date","Property_recorded_under_Block_No","Property_recorded_under_Parcel_No","Parcel_Cat_No","Parcel_Area","Nature_of_Entry","Tax_LP","Tax_Mils"]
header = " | ".join(["#"] + [c.split("_")[-1][:6] for c in preview_cols])
sep    = " | ".join(["---"] * (len(preview_cols) + 1))
lines  = [header, sep]
for i, r in enumerate(rows, 1):
    lines.append(" | ".join([str(i)] + [(r.get(c, "") or "")[:10] for c in preview_cols]))
display(Markdown("\n".join(lines)))


In [ ]:
# Patch the uploaded Hadita_{N}.xml — same geometry, only <Unicode> text overwritten.
# Matches patch_xml_text.py:patch_xml convention: replace the FIRST <Unicode>…</Unicode>
# inside each cell_r{R}_c{C} block. Cells the model didn't fill are overwritten with "".

from pathlib import Path

def _escape_xml(text: str) -> str:
    return text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

def patch_xml(xml_str: str, text_rows: list[dict]) -> tuple[str, int, int]:
    n_patched, n_skipped = 0, 0
    for r_idx, row in enumerate(text_rows):
        for c_idx, col in enumerate(LEFT_COLS):
            text = (row.get(col, "") or "").strip()
            cell_id = f'id="cell_r{r_idx}_c{c_idx}"'
            cell_start = xml_str.find(cell_id)
            if cell_start == -1:
                n_skipped += 1
                continue
            next_cell = xml_str.find("<TableCell", cell_start + len(cell_id))
            search_end = next_cell if next_cell != -1 else len(xml_str)
            u_start = xml_str.find("<Unicode>",  cell_start, search_end)
            u_end   = xml_str.find("</Unicode>", cell_start, search_end)
            if u_start == -1 or u_end == -1:
                n_skipped += 1
                continue
            content_start = u_start + len("<Unicode>")
            xml_str = xml_str[:content_start] + _escape_xml(text) + xml_str[u_end:]
            n_patched += 1
    return xml_str, n_patched, n_skipped

src_xml = Path(XML_PATH).read_text(encoding="utf-8")
patched, n_patched, n_skipped = patch_xml(src_xml, rows)

OUT_XML_PATH = f"Hadita_{PAGE_NUM}_G3.xml"
Path(OUT_XML_PATH).write_text(patched, encoding="utf-8")

# Diagnostic: how many (r,c) pairs the XML actually defines vs how many we filled
import re
xml_cells = set(re.findall(r'cell_r(\d+)_c(\d+)', src_xml))
xml_max_row = max((int(r) for r, _ in xml_cells), default=-1)
xml_max_col = max((int(c) for _, c in xml_cells), default=-1)
print(f"XML grid:  {xml_max_row + 1} rows × {xml_max_col + 1} cols  ({len(xml_cells)} cells defined)")
print(f"Model:     {len(rows)} rows × {len(LEFT_COLS)} cols")
print(f"Patched:   {n_patched} cells   ·   Skipped (no such cell or no <Unicode>): {n_skipped}")
print(f"wrote {OUT_XML_PATH}")

if len(rows) > xml_max_row + 1:
    print(f"\n⚠  Model produced more rows than the XML has slots — the last {len(rows) - (xml_max_row + 1)} row(s) "
          f"won't be patched. Check the trace for a phantom blank row, or this page genuinely has more rows than the geometry caught.")
elif len(rows) < xml_max_row + 1:
    print(f"\nℹ  Model produced fewer rows than the XML has slots — the bottom {xml_max_row + 1 - len(rows)} row(s) "
          f"of the XML were overwritten with \"\" (per the 'overwrite' policy).")


In [ ]:
from google.colab import files
files.download(OUT_XML_PATH)
files.download(ROWS_JSON_PATH)
try:
    files.download(RUNS_CSV)   # cumulative cost log; re-upload at the start of a fresh session
except Exception:
    pass
print("Downloaded the patched XML, raw rows JSON, and the cumulative g3_runs.csv.")
print(f"Next step: drop {OUT_XML_PATH} into Transkribus upload/final/  (replacing the existing Hadita_{PAGE_NUM}.xml),")
print( "or upload it directly into the Transkribus collection for RA correction.")


## Cost summary + projection

Run this cell any time to see per-page actuals and a projection for the next batch
based on what you've actually paid so far this session. If you resume in a fresh
Colab session, upload your saved `g3_runs.csv` first (any cell — `files.upload()`)
so the projection includes earlier runs.


In [ ]:
import csv
from pathlib import Path

RUNS_CSV = "g3_runs.csv"
if not Path(RUNS_CSV).exists():
    print(f"{RUNS_CSV} not found — run cell 7 on at least one page first.")
else:
    rows_log = list(csv.DictReader(open(RUNS_CSV)))
    if not rows_log:
        print(f"{RUNS_CSV} is empty.")
    else:
        print(f"{'page':>4} {'rows':>4} {'zoom':>4} {'elapsed':>8} {'in_tok':>7} {'out_tok':>7} {'tho_tok':>7} {'tool_tok':>8} {'cost':>8}")
        for r in rows_log:
            print(f"{r['page']:>4} {r['rows']:>4} {r['zoom_rounds']:>4} {float(r['elapsed_s']):>7.1f}s "
                  f"{int(r['input_tok']):>7,} {int(r['output_tok']):>7,} {int(r['thought_tok']):>7,} "
                  f"{int(r['tool_use_tok']):>8,} ${float(r['cost_usd']):>6.4f}")

        costs = [float(r["cost_usd"]) for r in rows_log]
        elapsed = [float(r["elapsed_s"]) for r in rows_log]
        n = len(costs)
        total = sum(costs); avg = total / n; med = sorted(costs)[n // 2]
        cmin, cmax = min(costs), max(costs)
        avg_s = sum(elapsed) / n

        print()
        print(f"Across {n} run(s):")
        print(f"  total cost so far : ${total:.4f}")
        print(f"  avg  per page     : ${avg:.4f}   (median ${med:.4f}, min ${cmin:.4f}, max ${cmax:.4f})")
        print(f"  avg  elapsed      : {avg_s:.1f}s / page")
        print()
        print(f"Projection for the next 100 pages at current rate:")
        print(f"  min  est: ${cmin*100:>6.2f}")
        print(f"  median  : ${med *100:>6.2f}")
        print(f"  mean    : ${avg *100:>6.2f}")
        print(f"  max  est: ${cmax*100:>6.2f}")
        print(f"  wall time at median elapsed (sequential): ~{avg_s*100/60:.0f} minutes")
